# 02 – Checks, cleaning & monthly demand

**Part A – three checks** (results decide the settings in `src/config.py`):
1. Where does normal activity end? → reference date
2. Are safety stock, lead time, price etc. constant per article (master-data snapshot)?
3. What do the inventory articles without any sale/usage do?
4. Bonus: is `inventory_level` the stock *before* or *after* the transaction?

**Part B – cleaning:** cut at reference date, keep inventory articles, flag & cap outliers,
build the article table and the monthly demand table (2019–2021).

**Output to share:** `outputs/tables/02_cleaning_summary.txt`

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from src import config as C

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

summary = []
def log(title, obj=""):
    text = obj.to_string() if isinstance(obj, (pd.DataFrame, pd.Series)) else str(obj)
    block = f"\n=== {title} ===\n{text}"
    print(block)
    summary.append(block)

df = pd.read_parquet(C.TRANSACTIONS_PARQUET)
df["date"] = pd.to_datetime(df["date"])
if "product_group" in df.columns and "product_category" not in df.columns:
    df = df.rename(columns={"product_group": "product_category"})
df["_row"] = np.arange(len(df))  # original file order, used for same-day ordering

# Inventory articles = safety_stock > 0 (latest value)
latest = df.sort_values("date", kind="stable").groupby("article_nr").tail(1).set_index("article_nr")
inv_articles = latest.index[latest["safety_stock"] > 0]
is_inv = df["article_nr"].isin(inv_articles)
log("Loaded", f"{len(df):,} rows | {df['article_nr'].nunique():,} articles | {len(inv_articles):,} inventory articles")


=== Loaded ===
2,694,849 rows | 58,764 articles | 1,751 inventory articles


## A1. Where does normal activity end?

In [2]:
REF = pd.Timestamp(C.REFERENCE_DATE)

monthly_counts = (df[df["date"] >= "2021-07-01"]
                  .groupby(df["date"].dt.to_period("M")).size())
log("Transactions per month from 2021-07", monthly_counts)

after = df[df["date"] > REF]
log(f"Transactions after reference date {REF.date()}", pd.Series({
    "rows": len(after),
    "articles": after["article_nr"].nunique(),
    "of which inventory articles": after.loc[after["article_nr"].isin(inv_articles), "article_nr"].nunique(),
}))
if len(after):
    log("…by date (top 15)", after["date"].dt.date.value_counts().head(15))
    log("…by entry type", after["entry_type"].map(C.ENTRY_TYPES).value_counts())


=== Transactions per month from 2021-07 ===
date
2021-07    25413
2021-08    24430
2021-09    26703
2021-10    23482
2021-11    22816
2021-12    18679
2022-01     7695
2022-03      587
2022-04      581
2022-05      816
2022-06      688
2022-07      483
2022-10      673
2022-11      453
2022-12      552
Freq: M

=== Transactions after reference date 2022-01-21 ===
rows                           4833
articles                       1264
of which inventory articles     620

=== …by date (top 15) ===
date
2022-05-01    816
2022-06-01    688
2022-10-01    673
2022-03-01    587
2022-04-01    581
2022-12-01    552
2022-07-01    483
2022-11-01    453

=== …by entry type ===
entry_type
Usage for production       2700
Sale                        892
Internal transaction        672
Ingoing from production     320
Procurement                 139
Outgoing adjustment          85
Ingoing adjustment           25


## A2. Are the attributes constant per article?
Share of articles where an attribute takes more than one value over all their transactions.
If ~0%, the attribute is a master-data snapshot copied onto every row.

In [3]:
attr_cols = ["safety_stock", "lead_time", "moq", "last_direct_cost", "vendor_nr", "base_unit",
             "eshop", "submarket", "item_main_group", "item_sub_group", "item_category", "product_category"]
attr_cols = [c for c in attr_cols if c in df.columns]

nun = df.groupby("article_nr")[attr_cols].nunique()
const_check = pd.DataFrame({
    "all articles: % with >1 value": (nun.gt(1).mean() * 100).round(2),
    "inventory articles: % with >1 value": (nun.loc[inv_articles].gt(1).mean() * 100).round(2),
})
log("Attribute variation per article", const_check)


=== Attribute variation per article ===
                  all articles: % with >1 value  inventory articles: % with >1 value
safety_stock                               0.00                                 0.00
lead_time                                  0.00                                 0.00
moq                                        0.00                                 0.00
last_direct_cost                           0.00                                 0.00
vendor_nr                                  0.00                                 0.00
base_unit                                  0.00                                 0.00
eshop                                      0.00                                 0.00
submarket                                  0.00                                 0.00
item_main_group                            0.00                                 0.00
item_sub_group                             0.00                                 0.00
item_category           

## A3. Inventory articles without any sale/usage

In [4]:
inv_df = df[is_inv]
has_demand = inv_df.loc[inv_df["entry_type"].isin(C.DEMAND_TYPES), "article_nr"].unique()
no_demand = pd.Index(inv_articles).difference(has_demand)

nd = inv_df[inv_df["article_nr"].isin(no_demand)]
log(f"No-demand inventory articles: {len(no_demand)} – their transactions by entry type",
    nd["entry_type"].map(C.ENTRY_TYPES).value_counts())

nd_art = nd.groupby("article_nr").agg(first=("date", "min"), last=("date", "max"), n_tx=("date", "size"))
nd_art = nd_art.join(latest[["inventory_level", "last_direct_cost", "safety_stock", "lead_time"]])
nd_art["stock_value"] = nd_art["inventory_level"].clip(lower=0) * nd_art["last_direct_cost"]
log("No-demand articles – first transaction year", nd_art["first"].dt.year.value_counts().sort_index())
log("No-demand articles – describe", nd_art[["n_tx", "inventory_level", "stock_value", "safety_stock"]].describe().round(2))


=== No-demand inventory articles: 214 – their transactions by entry type ===
entry_type
Procurement                7985
Outgoing adjustment        1346
Internal transaction        730
Ingoing adjustment          117
Ingoing from production       1

=== No-demand articles – first transaction year ===
first
2007    36
2008     8
2009     8
2010     5
2011     4
2012     4
2013    14
2014     2
2015     3
2016     9
2017     7
2018    11
2019    13
2020    15
2021    63
2022    12

=== No-demand articles – describe ===
         n_tx  inventory_level  stock_value  safety_stock
count  214.00           214.00       214.00        214.00
mean    47.57           512.73       500.37        284.15
std     89.10          3167.73      1303.17       1756.60
min      1.00            -3.33         0.00          1.00
25%      3.00             0.00         0.00          2.25
50%     10.50             9.44       110.44         22.00
75%     51.75           197.71       439.18        111.00
max    561.00

## A4. Is `inventory_level` before or after the transaction?
From the raw lines it looks like *before*: next row's `inventory_level` = this row's `inventory_level + quantity`.
Same-day order is unknown, so we also report the match rate only where the next transaction is on a later day.

In [5]:
s = inv_df.sort_values(["article_nr", "date", "_row"], ascending=[True, True, False])  # file looks newest-first
s = s.assign(after_tx=s["inventory_level"] + s["quantity"])
nxt_level = s.groupby("article_nr")["inventory_level"].shift(-1)
nxt_date = s.groupby("article_nr")["date"].shift(-1)
valid = nxt_level.notna()
match_before = np.isclose(s["after_tx"], nxt_level, atol=0.01) & valid
match_after  = np.isclose(s["inventory_level"], nxt_level - s.groupby("article_nr")["quantity"].shift(-1), atol=0.01) & valid
diff_day = valid & (nxt_date > s["date"])
log("inventory_level consistency", pd.Series({
    "H1 'before': next level = level + qty  (all rows)": f"{match_before[valid].mean():.1%}",
    "H1 'before': (next tx on later day)": f"{match_before[diff_day].mean():.1%}",
    "H2 'after':  next level = level + next qty (all rows)": f"{match_after[valid].mean():.1%}",
}))


=== inventory_level consistency ===
H1 'before': next level = level + qty  (all rows)        85.1%
H1 'before': (next tx on later day)                      49.0%
H2 'after':  next level = level + next qty (all rows)    22.0%


## B. Cleaning decisions
Check the outputs above. If needed, change `REFERENCE_DATE` / window in `src/config.py` and re-run.

In [6]:
start, end = pd.Timestamp(C.ANALYSIS_START), pd.Timestamp(C.ANALYSIS_END)

tx = df[is_inv & (df["date"] <= REF)].copy()
log("Kept transactions", f"{len(tx):,} rows of inventory articles up to {REF.date()} "
                         f"(dropped {int((is_inv & (df['date'] > REF)).sum()):,} later rows)")

# Stock after each transaction (based on A4: inventory_level = stock before)
tx["stock_after"] = tx["inventory_level"] + tx["quantity"]

# Demand: outflows of sales + production usage are negative → flip sign; returns become negative demand
tx["is_demand"] = tx["entry_type"].isin(C.DEMAND_TYPES)
tx["demand"] = np.where(tx["is_demand"], -tx["quantity"], 0.0)


=== Kept transactions ===
1,610,914 rows of inventory articles up to 2022-01-21 (dropped 3,225 later rows)


### Outliers
Only **clear one-off spikes** are capped: a demand transaction counts as an outlier if it is
**> 10x the article's median** transaction **and no other transaction of that article reaches 50% of its size**.
Recurring large orders (lumpy B2B demand) are kept – they are real demand.
The outlier is capped at max(10x median, second-largest transaction). Only articles with >= 10 demand transactions.

In [7]:
win = tx["is_demand"] & tx["date"].between(start, end) & (tx["demand"] > 0)
w_d = tx.loc[win, ["article_nr", "demand"]]

top2 = (w_d.sort_values("demand", ascending=False).groupby("article_nr")["demand"]
          .apply(lambda x: pd.Series({"largest": x.iloc[0], "second": x.iloc[1] if len(x) > 1 else 0.0}))
          .unstack())
stats = w_d.groupby("article_nr")["demand"].agg(n="size", median="median").join(top2)
stats["is_spike_article"] = ((stats["n"] >= C.OUTLIER_MIN_TRANSACTIONS)
                             & (stats["largest"] > C.OUTLIER_MEDIAN_FACTOR * stats["median"])
                             & (stats["second"] < C.OUTLIER_ISOLATION * stats["largest"]))
stats["cap"] = np.maximum(C.OUTLIER_MEDIAN_FACTOR * stats["median"], stats["second"])

tx = tx.merge(stats[["largest", "cap", "is_spike_article"]], left_on="article_nr", right_index=True, how="left")
tx["is_outlier"] = win & tx["is_spike_article"].fillna(False).astype(bool) & np.isclose(tx["demand"], tx["largest"])
tx["demand_capped"] = np.where(tx["is_outlier"], tx["cap"], tx["demand"])
tx = tx.drop(columns=["largest", "is_spike_article"])

out = tx[tx["is_outlier"]]
log("Outliers in demand window", pd.Series({
    "outlier transactions": len(out),
    "articles affected": out["article_nr"].nunique(),
    "demand volume removed by capping (% of window demand)":
        f"{(out['demand'] - out['demand_capped']).sum() / tx.loc[win, 'demand'].sum():.2%}",
}))
log("Largest outliers (top 10)",
    out.nlargest(10, "demand")[["article_nr", "date", "entry_type", "demand", "cap"]])


=== Outliers in demand window ===
outlier transactions                                       134
articles affected                                          134
demand volume removed by capping (% of window demand)    0.16%

=== Largest outliers (top 10) ===
        article_nr       date  entry_type        demand           cap
2527637    1333081 2021-10-27           1  17771.992268  11107.495168
2521269    1333887 2021-12-10           1  15994.793041  13328.994201
911458     1418959 2021-09-30           1   8230.653919   3998.698260
634782     1420449 2021-03-17           5   6753.357062   1688.339265
758969     1419630 2021-11-15           1   5553.747584   2221.499034
1112592    1412486 2020-01-17           5   5553.747584   2665.798840
2526094    1333090 2021-03-19           1   4276.385640   3887.623309
746781     1419796 2019-03-21           5   3498.860978   1732.769246
648425     1420327 2021-08-10           5   3186.740364   1577.264314
1075544    1412556 2021-04-10           1

### Article table
One row per inventory article: master data (latest values), history, data-quality flags.

In [8]:
last_demand = tx[tx["is_demand"] & (tx["demand"] > 0)].groupby("article_nr")["date"].max()
first_tx = df[is_inv].groupby("article_nr")["date"].min()
neg_stock = tx.groupby("article_nr")["stock_after"].min() < 0

art = latest.loc[inv_articles, [c for c in attr_cols + ["inventory_level"] if c in latest.columns]].copy()
art = art.rename(columns={"inventory_level": "inventory_level_latest_row"})
art["first_tx"] = first_tx
art["last_demand"] = last_demand
art["days_since_demand"] = (REF - art["last_demand"]).dt.days
art["flag_no_demand"] = art["last_demand"].isna()
art["flag_lead_time_missing"] = art["lead_time"] == 0
art["flag_price_zero"] = art["last_direct_cost"] <= 0
art["flag_moq_missing"] = art["moq"] == 0
art["flag_negative_stock"] = neg_stock.reindex(art.index).fillna(False)
art["flag_new_in_window"] = art["first_tx"] > start
art["n_outliers_capped"] = out.groupby("article_nr").size().reindex(art.index).fillna(0).astype(int)

flags = [c for c in art.columns if c.startswith("flag_")]
log("Data-quality flags (number of inventory articles)", art[flags].sum())


=== Data-quality flags (number of inventory articles) ===
flag_no_demand            215
flag_lead_time_missing     37
flag_price_zero             4
flag_moq_missing          462
flag_negative_stock       712
flag_new_in_window        378


### Monthly demand table (2019-01 → 2021-12)
Complete article × month grid with zeros for months without demand – essential for a correct
standard deviation. Months before an article's first transaction stay empty (NaN), so new articles
aren't penalised. Net monthly demand below 0 (more returns than sales) is set to 0.

In [9]:
months = pd.period_range(start, end, freq="M")
w = tx[tx["is_demand"] & tx["date"].between(start, end)].copy()
w["month"] = w["date"].dt.to_period("M")
agg = w.groupby(["article_nr", "month"])[["demand", "demand_capped"]].sum()

grid = pd.MultiIndex.from_product([art.index, months], names=["article_nr", "month"])
dm = agg.reindex(grid, fill_value=0.0).clip(lower=0).rename(
        columns={"demand": "demand_raw", "demand_capped": "demand_capped"}).reset_index()

first_month = art["first_tx"].dt.to_period("M")
before_intro = dm["month"] < dm["article_nr"].map(first_month)
dm.loc[before_intro, ["demand_raw", "demand_capped"]] = np.nan

per_art = dm.groupby("article_nr")["demand_capped"].agg(
    months_observed="count", months_with_demand=lambda x: (x > 0).sum(), total="sum")
log("Monthly demand table", f"{dm['article_nr'].nunique():,} articles × {len(months)} months = {len(dm):,} rows")
log("Months with demand per article (distribution)",
    pd.cut(per_art["months_with_demand"], [-1, 0, 6, 12, 24, 35, 36],
           labels=["0", "1-6", "7-12", "13-24", "25-35", "36 (every month)"]).value_counts().sort_index())


=== Monthly demand table ===
1,751 articles × 36 months = 63,036 rows

=== Months with demand per article (distribution) ===
months_with_demand
0                   253
1-6                 273
7-12                214
13-24               355
25-35               421
36 (every month)    235


## Save

In [10]:
dm["month"] = dm["month"].dt.to_timestamp()
keep_cols = ["article_nr", "date", "_row", "entry_type", "quantity", "inventory_level", "stock_after",
             "demand", "demand_capped", "is_outlier", "last_direct_cost"]
tx[keep_cols].to_parquet(C.TRANSACTIONS_CLEAN, index=False)
art.reset_index().to_parquet(C.ARTICLES, index=False)
dm.to_parquet(C.DEMAND_MONTHLY, index=False)

for p in (C.TRANSACTIONS_CLEAN, C.ARTICLES, C.DEMAND_MONTHLY):
    log("Saved", f"{p.name} | {p.stat().st_size / 1e6:.1f} MB")

out_file = C.TABLES / "02_cleaning_summary.txt"
out_file.write_text("\n".join(summary), encoding="utf-8")
print(f"Summary written to {out_file} – share this file.")


=== Saved ===
transactions_clean.parquet | 28.7 MB

=== Saved ===
articles.parquet | 0.1 MB

=== Saved ===
demand_monthly.parquet | 0.2 MB
Summary written to /Users/nikolaosgeorgantopoulos/Repositories/inventory-optimization/outputs/tables/02_cleaning_summary.txt – share this file.
